In [3]:
%pip install pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


# PerfectPetClub — Customer Churn ETL Pipeline

**Assessment task:** create a customer-level dataset suitable for analysing subscription churn.

This notebook implements the Extract, Transform and Load stages using Python, Pandas and SQLite. It reads the five supplied project files, validates their structure and quality, creates customer-level behavioural features, and writes a Parquet dataset.

## Churn target definition

The assessment defines churn as *cancelling the auto-renewal of a subscription*. In this dataset, `subscription.auto_renew == 0` is therefore used as the positive churn label (`churn = 1`); `auto_renew == 1` becomes `churn = 0`.

This is a **cross-sectional classification dataset**, not proof that the model predicts a future cancellation event. The supplied data does not provide a timestamp for when auto-renew was switched off. Features are aggregated from the supplied activity records, so temporal leakage cannot be ruled out completely. A production forecasting model would need a fixed prediction date and only features recorded before that date.

## Input files

Place the files in `data/raw/` relative to the project root:

- `PerfectPetClub.db` — SQLite sales, customer, subscription, product and review tables.
- `PerfectPetClub_CRM_report.csv` — customer service contacts.
- `PerfectPetClub_server_log.csv` — web application login records.
- `PerfectPetClub_survey.csv` — customer survey responses.
- `PerfectPetClub_survey_questions.txt` — survey field descriptions (reference only).

## Output

`data/processed/customer_churn.parquet`

The output has one row per customer. `customer_id` is retained as a record key but must **not** be used as a predictive feature. Names, addresses, email addresses, password hashes, IP addresses, session IDs and transaction references are not included in the modelling dataset.

In [4]:
%pip install pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import sqlite3

print("ETL setup successful!")

ETL setup successful!


In [6]:
%pip install pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

print("Current folder:", PROJECT_ROOT)
print("Raw data folder exists:", (PROJECT_ROOT / "data" / "raw").exists())
print("Database exists:", (PROJECT_ROOT / "data" / "raw" / "PerfectPetClub.db").exists())

Current folder: c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project\notebooks
Raw data folder exists: False
Database exists: False


In [15]:
from pathlib import Path

# Move one level up from notebooks to the project folder
PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Project folder:", PROJECT_ROOT)
print("Raw data folder exists:", RAW_DATA.exists())
print("Database exists:", (RAW_DATA / "PerfectPetClub.db").exists())

Project folder: c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project
Raw data folder exists: True
Database exists: True


In [16]:
import sqlite3
import pandas as pd

# Connect to the PerfectPetClub database
db_path = RAW_DATA / "PerfectPetClub.db"

connection = sqlite3.connect(db_path)

print("Connected to PerfectPetClub database!")

Connected to PerfectPetClub database!


## 1. Import libraries and configure paths

**What this block does:** imports the libraries used for SQL extraction, data manipulation, validation, logging and Parquet output. Paths are built relative to the notebook location, so the project can be moved without rewriting absolute paths.

In [17]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

print("Tables available in the database:")
display(tables)

Tables available in the database:


,name
0,payment
1,sqlite_sequence
2,product_category
3,product
4,subscription_type
5,subscription
6,customer
7,sale
8,sale_product
9,product_review


In [8]:
from pathlib import Path
import logging
import sqlite3
from typing import Dict

import numpy as np
import pandas as pd

# The notebook is in <project>/notebooks; the project root is its parent.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = RAW_DIR / "PerfectPetClub.db"
CRM_PATH = RAW_DIR / "PerfectPetClub_CRM_report.csv"
SERVER_LOG_PATH = RAW_DIR / "PerfectPetClub_server_log.csv"
SURVEY_PATH = RAW_DIR / "PerfectPetClub_survey.csv"
SURVEY_QUESTIONS_PATH = RAW_DIR / "PerfectPetClub_survey_questions.txt"
OUTPUT_PATH = PROCESSED_DIR / "customer_churn.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("perfectpetclub_etl")

required_paths = [DB_PATH, CRM_PATH, SERVER_LOG_PATH, SURVEY_PATH, SURVEY_QUESTIONS_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "The following required input files are missing:\n" + "\n".join(missing_paths)
    )

print("Project root:", PROJECT_ROOT)
print("All five input files are present.")

Project root: c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project
All five input files are present.


In [18]:
import sqlite3
import pandas as pd

db_path = RAW_DATA / "PerfectPetClub.db"

connection = sqlite3.connect(db_path)

print("Connected to PerfectPetClub database!")

Connected to PerfectPetClub database!


## 2. Extract — inspect the SQLite database and read the source files

**What this block does:** first lists the SQLite tables and their row counts. The `extract_data` function then reads the relational tables and three CSV datasets into Pandas DataFrames. The survey-question text file is read as documentation, not as customer data.

**Why use functions?** Keeping extraction in one function makes the pipeline reusable and easier to test. It also separates reading data from cleaning and feature engineering.

In [19]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

display(tables)

,name
0,payment
1,sqlite_sequence
2,product_category
3,product
4,subscription_type
5,subscription
6,customer
7,sale
8,sale_product
9,product_review


In [9]:
def inspect_sqlite_database(db_path: Path) -> pd.DataFrame:
    """Return the table names and row counts in the supplied SQLite database."""
    with sqlite3.connect(db_path) as connection:
        table_names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'",
            connection
        )["name"].tolist()

        inventory = []
        for table_name in table_names:
            # Table names come from SQLite metadata, not user input.
            row_count = pd.read_sql_query(
                f'SELECT COUNT(*) AS row_count FROM "{table_name}"',
                connection
            ).loc[0, "row_count"]
            inventory.append({"table_name": table_name, "row_count": int(row_count)})

    return pd.DataFrame(inventory).sort_values("table_name").reset_index(drop=True)


def extract_data(
    db_path: Path,
    crm_path: Path,
    server_log_path: Path,
    survey_path: Path,
    survey_questions_path: Path
) -> Dict[str, object]:
    """Extract relational tables, CSV files and survey documentation."""
    logger.info("Starting extraction")

    with sqlite3.connect(db_path) as connection:
        table_names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'",
            connection
        )["name"].tolist()
        database_tables = {
            table_name: pd.read_sql_query(f'SELECT * FROM "{table_name}"', connection)
            for table_name in table_names
        }

    crm = pd.read_csv(crm_path)
    server_logs = pd.read_csv(server_log_path)
    survey = pd.read_csv(survey_path)
    survey_questions = survey_questions_path.read_text(encoding="utf-8")

    extracted = {
        "database": database_tables,
        "crm": crm,
        "server_logs": server_logs,
        "survey": survey,
        "survey_questions": survey_questions
    }

    logger.info(
        "Extracted %d database tables, %d CRM rows, %d server-log rows and %d survey rows",
        len(database_tables), len(crm), len(server_logs), len(survey)
    )
    return extracted


database_inventory = inspect_sqlite_database(DB_PATH)
display(database_inventory)

raw = extract_data(
    DB_PATH, CRM_PATH, SERVER_LOG_PATH, SURVEY_PATH, SURVEY_QUESTIONS_PATH
)

# Display source shapes to confirm extraction.
for table_name, frame in raw["database"].items():
    print(f"SQLite table {table_name}: {frame.shape}")
print("CRM:", raw["crm"].shape)
print("Server logs:", raw["server_logs"].shape)
print("Survey:", raw["survey"].shape)
print("Survey-question reference:\n", raw["survey_questions"])

,table_name,row_count
0,customer,221
1,payment,23188
2,product,194
3,product_category,26
4,product_review,10990
5,sale,23188
6,sale_product,59759
7,subscription,221
8,subscription_type,3


2026-11-22 01:26:22,495 | INFO | Starting extraction
2026-11-22 01:26:22,735 | INFO | Extracted 9 database tables, 7585 CRM rows, 67249 server-log rows and 221 survey rows


SQLite table payment: (23188, 4)
SQLite table product_category: (26, 2)
SQLite table product: (194, 4)
SQLite table subscription_type: (3, 3)
SQLite table subscription: (221, 5)
SQLite table customer: (221, 9)
SQLite table sale: (23188, 6)
SQLite table sale_product: (59759, 4)
SQLite table product_review: (10990, 5)
CRM: (7585, 4)
Server logs: (67249, 5)
Survey: (221, 5)
Survey-question reference:
 WebApp Popup Survey Questions 
email = customers email from logged in session.
reported_income = Q1. "What income bracket are you in?" [drop-down selector]
nps_recommend = Q2. "Would you recommend us to a friend? (no 0 - 10 yes)" [rating slider 0-10]
satisfaction_score = Q3. "Would you recommend us to a friend? (no 0 - 10 yes)" [rating slider 0-10]
online_comparison_freq = Q4. "How much time do you spend online shopping and comparing prices?" [rating slider 0-10]




In [20]:
for table in tables["name"]:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total FROM "{table}"',
        connection
    ).iloc[0, 0]

    print(f"{table}: {count} records")

payment: 23188 records
sqlite_sequence: 9 records
product_category: 26 records
product: 194 records
subscription_type: 3 records
subscription: 221 records
customer: 221 records
sale: 23188 records
sale_product: 59759 records
product_review: 10990 records


In [21]:
# Display the first five customer records
display(customer.head())

NameError: name 'customer' is not defined

In [22]:
print(db_tables.keys())

NameError: name 'db_tables' is not defined

In [24]:
display(customers.head())

,id,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1
1,2,Leyla Mammadova,Mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2
2,3,Sana Shah,Sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3
3,4,Omar Farooq,Omar.Farooq4@example.com,b21911abfc36881eea1278dc19b3f6b1b652ec4c213c00...,22 High Street,Tonbridge,Kent,TN9 1LM,4
4,5,Alejandro Blanco,Alejandro357@somemail.co.uk,fadf0729d5b6518a26ba92b664122884d15c4f646eac03...,31 New Road,Radlett,Hertfordshire,WD7 8PQ,5


In [25]:
print("Customer columns:")
print(customers.columns.tolist())

Customer columns:
['id', 'name', 'email', 'password_hash', 'address_line_1', 'address_line_2', 'county', 'postcode', 'subscription_id']


In [26]:
print("Customer churn columns:")
print(customer_churn.columns.tolist())

print("\nDataset shape:")
print(customer_churn.shape)

Customer churn columns:
['customer_id', 'churn', 'subscription_type_name', 'delivery_service', 'subscription_age_days', 'order_count', 'total_payment_amount', 'average_payment_amount', 'total_items_purchased', 'average_items_per_order', 'review_count', 'average_review_rating', 'low_rating_share', 'crm_contact_count', 'crm_complaint_count', 'crm_query_count', 'crm_channel_count', 'login_count', 'active_login_days', 'unique_sessions', 'mobile_login_count', 'desktop_login_count', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']

Dataset shape:
(221, 25)


In [27]:
print("Churn distribution:")
print(customer_churn["churn"].value_counts())

print("\nMissing values:")
print(customer_churn.isnull().sum())

print("\nDuplicate customer IDs:")
print(customer_churn["customer_id"].duplicated().sum())

Churn distribution:
churn
0    135
1     86
Name: count, dtype: int64

Missing values:
customer_id                0
churn                      0
subscription_type_name     0
delivery_service           0
subscription_age_days      0
order_count                0
total_payment_amount       0
average_payment_amount     0
total_items_purchased      0
average_items_per_order    0
review_count               0
average_review_rating      8
low_rating_share           8
crm_contact_count          0
crm_complaint_count        0
crm_query_count            0
crm_channel_count          0
login_count                0
active_login_days          0
unique_sessions            0
mobile_login_count         0
desktop_login_count        0
nps_recommend              0
satisfaction_score         0
online_comparison_freq     0
dtype: int64

Duplicate customer IDs:
0


In [28]:
missing = customer_churn.isnull().sum()

print("Columns with missing values:")
print(missing[missing > 0])

Columns with missing values:
average_review_rating    8
low_rating_share         8
dtype: int64


In [29]:
customer_churn["average_review_rating"] = (
    customer_churn["average_review_rating"].fillna(0)
)

customer_churn["low_rating_share"] = (
    customer_churn["low_rating_share"].fillna(0)
)

print("Remaining missing values:", customer_churn.isnull().sum().sum())

Remaining missing values: 0


In [30]:
OUTPUT_PATH = PROCESSED_DATA / "customer_churn.parquet"

customer_churn.to_parquet(OUTPUT_PATH, index=False)

print("Dataset saved successfully!")
print("File location:", OUTPUT_PATH)
print("Dataset shape:", customer_churn.shape)

Dataset saved successfully!
File location: c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project\data\processed\customer_churn.parquet
Dataset shape: (221, 25)


In [31]:
check = pd.read_parquet(OUTPUT_PATH)

print("Parquet file loaded successfully!")
print("Shape:", check.shape)
print("Missing values:", check.isnull().sum().sum())

Parquet file loaded successfully!
Shape: (221, 25)
Missing values: 0


## 3. Data quality checks — inspect source structure

**What this block does:** checks required tables and columns, reports missing values, duplicate rows and data types, and checks key uniqueness. These checks are deliberately visible: they provide evidence that assumptions are being tested rather than silently accepted.

**Important source-specific observations to verify when running:**
- The customer and survey tables should have unique customer email addresses.
- CRM records are linked by `customer_id`.
- Server logs and survey records are linked to customers by email.
- The server log contains technical fields such as IP and session ID; these are used only for aggregation and are not retained in the final dataset.

## 3. Data quality checks — inspect source structure

**What this block does:** checks required tables and columns, reports missing values, duplicate rows and data types, and checks key uniqueness. These checks are deliberately visible: they provide evidence that assumptions are being tested rather than silently accepted.

**Important source-specific observations to verify when running:**
- The customer and survey tables should have unique customer email addresses.
- CRM records are linked by `customer_id`.
- Server logs and survey records are linked to customers by email.
- The server log contains technical fields such as IP and session ID; these are used only for aggregation and are not retained in the final dataset.

In [10]:
REQUIRED_DB_TABLES = {
    "customer", "subscription", "subscription_type", "sale", "payment",
    "sale_product", "product", "product_category", "product_review"
}
REQUIRED_COLUMNS = {
    "customer": {"id", "email", "subscription_id"},
    "subscription": {"id", "start_date", "end_date", "auto_renew", "subscription_type_id"},
    "subscription_type": {"id", "name", "delivery_service"},
    "sale": {"id", "order_date", "customer_id", "payment_id"},
    "payment": {"id", "amount"},
    "sale_product": {"quantity", "sale_id", "product_id"},
    "product_review": {"star_rating", "customer_id", "sale_id"}
}


def validate_source_structure(data: Dict[str, object]) -> None:
    """Fail early if a required source table or column is absent."""
    tables = data["database"]
    missing_tables = REQUIRED_DB_TABLES - set(tables)
    if missing_tables:
        raise ValueError(f"Missing required SQLite tables: {sorted(missing_tables)}")

    for table_name, required_columns in REQUIRED_COLUMNS.items():
        missing_columns = required_columns - set(tables[table_name].columns)
        if missing_columns:
            raise ValueError(
                f"Table {table_name} is missing columns: {sorted(missing_columns)}"
            )

    required_csv_columns = {
        "crm": {"customer_id", "recieved_datatime", "platform", "type"},
        "server_logs": {"datetime", "session_id", "ip", "login", "user_agent"},
        "survey": {
            "email", "reported_income", "nps_recommend",
            "satisfaction_score", "online_comparison_freq"
        }
    }
    for source_name, required_columns in required_csv_columns.items():
        missing_columns = required_columns - set(data[source_name].columns)
        if missing_columns:
            raise ValueError(
                f"{source_name} is missing columns: {sorted(missing_columns)}"
            )


def source_quality_report(data: Dict[str, object]) -> pd.DataFrame:
    """Summarise row count, duplicate rows and missing cells for each source."""
    reports = []
    frames = list(data["database"].items()) + [
        ("crm", data["crm"]),
        ("server_logs", data["server_logs"]),
        ("survey", data["survey"])
    ]
    for name, frame in frames:
        reports.append({
            "source": name,
            "rows": len(frame),
            "columns": len(frame.columns),
            "duplicate_rows": int(frame.duplicated().sum()),
            "missing_cells": int(frame.isna().sum().sum())
        })
    return pd.DataFrame(reports)


validate_source_structure(raw)
quality_report = source_quality_report(raw)
display(quality_report)

# Key-level checks.
customers = raw["database"]["customer"]
survey = raw["survey"]
assert customers["id"].is_unique, "Customer IDs must be unique."
assert customers["email"].str.lower().is_unique, "Customer emails must be unique."
assert survey["email"].str.lower().is_unique, "Survey email should be unique per customer."
assert set(raw["database"]["subscription"]["auto_renew"].dropna().unique()).issubset({0, 1}), (
    "auto_renew must contain only 0/1 values."
)
print("Core structure and key checks passed.")

,source,rows,columns,duplicate_rows,missing_cells
0,payment,23188,4,0,0
1,product_category,26,2,0,0
2,product,194,4,0,3
3,subscription_type,3,3,0,0
4,subscription,221,5,0,0
5,customer,221,9,0,0
6,sale,23188,6,0,2180
7,sale_product,59759,4,0,0
8,product_review,10990,5,0,0
9,crm,7585,4,0,0


Core structure and key checks passed.


## 4. Transform — standardise and aggregate the data to one row per customer

**What this block does:** standardises join keys and dates, then creates customer-level aggregates from sales, purchased quantities, product reviews, CRM contacts and website logins. It joins these features to the customer/subscription record and survey responses.

**Feature engineering choices:**
- Sales: number of orders, total recorded payment amount, average payment amount, total items and average items per order.
- Reviews: review count, average star rating and proportion of low ratings (1–2 stars).
- CRM: total contacts, complaint count, query count and number of contact channels.
- Web activity: login count, active login days, unique sessions and desktop/mobile login counts.
- Survey: recommendation, satisfaction and online price-comparison scores.

**Privacy and modelling choices:** customer name, address, postcode, email, password hash, IP address, session ID, user-agent text and transaction reference are not retained. `customer_id` is kept only as a record key and must be dropped before model training. `reported_income` is not included in the modelling dataset because it is demographic information and the assignment notes that demographic patterns were not intentionally generated. The survey-question document also describes `satisfaction_score` with wording similar to the recommendation question; this should be acknowledged as a source-documentation ambiguity.

In [11]:
def _normalise_email(series: pd.Series) -> pd.Series:
    """Normalise email join keys without retaining them in the final output."""
    return series.astype("string").str.strip().str.lower()


def _to_datetime(series: pd.Series) -> pd.Series:
    """Parse source timestamps; invalid values become NaT for quality reporting."""
    return pd.to_datetime(series, errors="coerce")


def build_sales_features(tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Aggregate order, payment and quantity information to customer level."""
    sales = tables["sale"].copy()
    payments = tables["payment"].copy()
    sale_products = tables["sale_product"].copy()

    sales["order_date"] = _to_datetime(sales["order_date"])
    sales = sales.merge(
        payments[["id", "amount"]].rename(columns={"id": "payment_id", "amount": "payment_amount"}),
        on="payment_id", how="left", validate="many_to_one"
    )

    quantity_by_sale = (
        sale_products.groupby("sale_id", as_index=False)
        .agg(total_items=("quantity", "sum"))
    )
    sales = sales.merge(
        quantity_by_sale, left_on="id", right_on="sale_id",
        how="left", validate="one_to_one"
    )
    sales["total_items"] = sales["total_items"].fillna(0)

    features = (
        sales.groupby("customer_id", as_index=False)
        .agg(
            order_count=("id", "nunique"),
            total_payment_amount=("payment_amount", "sum"),
            average_payment_amount=("payment_amount", "mean"),
            total_items_purchased=("total_items", "sum"),
            average_items_per_order=("total_items", "mean"),
            first_order_date=("order_date", "min"),
            last_order_date=("order_date", "max")
        )
    )
    return features


def build_review_features(tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Create customer-level product-review features."""
    reviews = tables["product_review"].copy()
    reviews["star_rating"] = pd.to_numeric(reviews["star_rating"], errors="coerce")
    return (
        reviews.groupby("customer_id", as_index=False)
        .agg(
            review_count=("star_rating", "count"),
            average_review_rating=("star_rating", "mean"),
            low_rating_count=("star_rating", lambda values: values.le(2).sum())
        )
        .assign(
            low_rating_share=lambda frame:
                frame["low_rating_count"] / frame["review_count"].replace(0, np.nan)
        )
        .drop(columns=["low_rating_count"])
    )


def build_crm_features(crm: pd.DataFrame) -> pd.DataFrame:
    """Aggregate customer service contacts by customer ID."""
    frame = crm.copy()
    frame["recieved_datatime"] = _to_datetime(frame["recieved_datatime"])
    frame["type"] = frame["type"].astype("string").str.strip().str.lower()
    frame["platform"] = frame["platform"].astype("string").str.strip().str.lower()

    grouped = frame.groupby("customer_id")
    result = grouped.agg(
        crm_contact_count=("customer_id", "size"),
        crm_complaint_count=("type", lambda values: values.eq("complaint").sum()),
        crm_query_count=("type", lambda values: values.eq("query").sum()),
        crm_channel_count=("platform", "nunique"),
        first_crm_contact=("recieved_datatime", "min"),
        last_crm_contact=("recieved_datatime", "max")
    ).reset_index()
    return result


def build_server_features(server_logs: pd.DataFrame, customer_lookup: pd.DataFrame) -> pd.DataFrame:
    """Map logins to customer IDs, then aggregate activity without retaining identifiers."""
    logs = server_logs.copy()
    logs["login_key"] = _normalise_email(logs["login"])
    logs["datetime"] = _to_datetime(logs["datetime"])
    logs["user_agent_clean"] = logs["user_agent"].astype("string").str.lower()

    lookup = customer_lookup[["customer_id", "email_key"]].copy()
    logs = logs.merge(
        lookup, left_on="login_key", right_on="email_key",
        how="left", validate="many_to_one"
    )

    # Device indicators are deliberately broad text-pattern features.
    logs["is_mobile_login"] = logs["user_agent_clean"].str.contains(
        "mobile", na=False
    ).astype(int)
    logs["is_desktop_login"] = logs["user_agent_clean"].str.contains(
        "desktop|laptop", regex=True, na=False
    ).astype(int)

    result = (
        logs.dropna(subset=["customer_id"])
        .groupby("customer_id", as_index=False)
        .agg(
            login_count=("session_id", "size"),
            active_login_days=("datetime", lambda values: values.dt.date.nunique()),
            unique_sessions=("session_id", "nunique"),
            mobile_login_count=("is_mobile_login", "sum"),
            desktop_login_count=("is_desktop_login", "sum"),
            first_login_datetime=("datetime", "min"),
            last_login_datetime=("datetime", "max")
        )
    )
    return result


def transform_data(data: Dict[str, object]) -> pd.DataFrame:
    """Clean, engineer and join all sources into one row per customer."""
    logger.info("Starting transformation")
    tables = data["database"]

    customer = tables["customer"].copy()
    subscription = tables["subscription"].copy()
    subscription_type = tables["subscription_type"].copy()

    # Keep only the customer ID and fields required for joins/features.
    customer["email_key"] = _normalise_email(customer["email"])
    customer_base = customer[["id", "email_key", "subscription_id"]].rename(
        columns={"id": "customer_id"}
    )

    subscription["start_date"] = _to_datetime(subscription["start_date"])
    subscription["end_date"] = _to_datetime(subscription["end_date"])
    subscription["auto_renew"] = pd.to_numeric(subscription["auto_renew"], errors="coerce")
    subscription["churn"] = subscription["auto_renew"].map({0: 1, 1: 0})

    # A subscription ID should map to exactly one subscription record.
    if not subscription["id"].is_unique:
        raise ValueError("Subscription IDs are not unique.")
    if subscription["churn"].isna().any():
        raise ValueError("Unexpected or missing auto_renew values; cannot define churn label.")

    subscription = subscription.merge(
        subscription_type.rename(columns={
            "id": "subscription_type_id",
            "name": "subscription_type_name",
            "delivery_service": "delivery_service"
        }),
        on="subscription_type_id", how="left", validate="many_to_one"
    )

    # Build a compact subscription frame; do not use end_date as a feature.
    subscription_features = subscription[[
        "id", "start_date", "auto_renew", "churn",
        "subscription_type_name", "delivery_service"
    ]].rename(columns={"id": "subscription_id"})

    customer_features = customer_base.merge(
        subscription_features, on="subscription_id",
        how="left", validate="many_to_one"
    )
    if customer_features["churn"].isna().any():
        raise ValueError("Some customers do not have a valid subscription/churn label.")

    # Derive subscription age at the latest observed activity date, not at today's date.
    activity_dates = []
    for source_name, date_col in [
        ("crm", "recieved_datatime"),
        ("server_logs", "datetime")
    ]:
        parsed = _to_datetime(data[source_name][date_col])
        if parsed.notna().any():
            activity_dates.append(parsed.max())

    for table_name, date_col in [("sale", "order_date")]:
        parsed = _to_datetime(tables[table_name][date_col])
        if parsed.notna().any():
            activity_dates.append(parsed.max())

    reference_date = max(activity_dates) if activity_dates else pd.Timestamp("2026-01-01")
    customer_features["subscription_age_days"] = (
        reference_date - customer_features["start_date"]
    ).dt.total_seconds() / 86400

    # Add customer-level aggregates. Left joins preserve customers with no activity.
    feature_frames = [
        build_sales_features(tables),
        build_review_features(tables),
        build_crm_features(data["crm"]),
        build_server_features(data["server_logs"], customer_base),
    ]
    for features in feature_frames:
        customer_features = customer_features.merge(
            features, on="customer_id", how="left", validate="one_to_one"
        )

    # Survey is linked by normalised email; email itself is not carried forward.
    survey = data["survey"].copy()
    survey["email_key"] = _normalise_email(survey["email"])
    survey = survey.drop(columns=["email", "reported_income"], errors="ignore")
    survey_numeric = ["nps_recommend", "satisfaction_score", "online_comparison_freq"]
    for column in survey_numeric:
        survey[column] = pd.to_numeric(survey[column], errors="coerce")
        invalid = survey[column].notna() & ~survey[column].between(0, 10)
        if invalid.any():
            raise ValueError(f"Survey field {column} contains values outside 0–10.")
    survey = survey.rename(columns={"email_key": "email_key"})
    customer_features = customer_features.merge(
        survey, on="email_key", how="left", validate="one_to_one"
    )

    # Drop join keys and source-specific raw dates after deriving useful features.
    customer_features = customer_features.drop(
        columns=[
            "subscription_id", "email_key", "start_date",
            "first_order_date", "last_order_date",
            "first_crm_contact", "last_crm_contact",
            "first_login_datetime", "last_login_datetime",
            "auto_renew"
        ],
        errors="ignore"
    )

    # Convert non-finite numeric values to missing; preserve missingness for imputation later.
    numeric_columns = customer_features.select_dtypes(include=[np.number]).columns
    customer_features[numeric_columns] = customer_features[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

    # Keep one row per customer, with the target as an integer 0/1.
    customer_features["churn"] = customer_features["churn"].astype("int8")
    customer_features = customer_features.sort_values("customer_id").reset_index(drop=True)

    logger.info(
        "Transformation completed: %d customers and %d columns",
        len(customer_features), len(customer_features.columns)
    )
    return customer_features


customer_churn = transform_data(raw)
print("Final shape:", customer_churn.shape)
display(customer_churn.head())

2026-11-22 01:26:22,932 | INFO | Starting transformation
2026-11-22 01:26:23,304 | INFO | Transformation completed: 221 customers and 25 columns


Final shape: (221, 25)


,customer_id,churn,subscription_type_name,delivery_service,subscription_age_days,order_count,total_payment_amount,average_payment_amount,total_items_purchased,average_items_per_order,...,crm_query_count,crm_channel_count,login_count,active_login_days,unique_sessions,mobile_login_count,desktop_login_count,nps_recommend,satisfaction_score,online_comparison_freq
0,1,1,SuperSaver,Economy72,313.728698,20,74455,3722.750000,46,2.300000,...,1,2.0,48,11,48,0,0,8,6,5
1,2,0,SuperSaver,Economy72,313.728698,253,1036961,4098.660079,748,2.956522,...,24,3.0,1085,295,1085,0,0,5,8,9
2,3,0,Premium,SignedFor24,313.728698,304,1284692,4225.960526,824,2.710526,...,77,3.0,624,266,624,0,0,9,5,5
3,4,0,Premium,SignedFor24,313.728698,1055,9577504,9078.202844,7362,6.978199,...,149,3.0,1540,309,1540,0,0,9,3,5
4,5,1,SuperSaver,Economy72,313.728698,44,179738,4084.954545,116,2.636364,...,4,3.0,388,99,388,388,0,0,8,10


## 5. Validate the transformed customer-level dataset

**What this block does:** verifies the grain (one row per customer), target definition, numeric survey ranges and feature validity. It reports missing values rather than automatically filling every missing feature with zero.

**Why not fill every missing value with zero?** A customer with no CRM contacts is different from a customer whose CRM data is unknown. For count features, a missing aggregate after a left join usually means no recorded activity and can reasonably be set to zero. For survey scores and average ratings, missing means no response/review and should remain missing so the ML preprocessing stage can impute it appropriately.

The code below applies zero only to activity-count features. Mean scores and rates remain null when no observation exists.

In [12]:
def validate_final_dataset(frame: pd.DataFrame) -> pd.DataFrame:
    """Run final integrity checks and return a missing-value report."""
    if frame.empty:
        raise ValueError("Final dataset is empty.")
    if "customer_id" not in frame or "churn" not in frame:
        raise ValueError("customer_id and churn columns are required.")
    if frame["customer_id"].duplicated().any():
        raise ValueError("More than one row exists for a customer.")
    if frame["churn"].isna().any() or not set(frame["churn"].unique()).issubset({0, 1}):
        raise ValueError("Churn target must be complete and binary (0/1).")

    # Counts from left-joined activity tables are zero when no event was recorded.
    count_columns = [
        "order_count", "total_items_purchased", "review_count",
        "crm_contact_count", "crm_complaint_count", "crm_query_count",
        "crm_channel_count", "login_count", "active_login_days",
        "unique_sessions", "mobile_login_count", "desktop_login_count"
    ]
    for column in count_columns:
        if column in frame.columns:
            frame[column] = frame[column].fillna(0)

    for column in [
        "nps_recommend", "satisfaction_score", "online_comparison_freq"
    ]:
        if column in frame.columns:
            invalid = frame[column].dropna().loc[
                ~frame[column].dropna().between(0, 10)
            ]
            if not invalid.empty:
                raise ValueError(f"{column} contains values outside 0–10.")

    if (frame.select_dtypes(include=[np.number]) < 0).any().any():
        # Negative values are not expected for counts, payment totals or durations.
        # The check is intentionally reported as a warning because some future
        # engineered numeric features might legitimately be negative.
        logger.warning("At least one numeric field contains negative values; inspect before modelling.")

    missing_report = (
        frame.isna().sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_percent=lambda x: (x["missing_count"] / len(frame) * 100).round(2))
        .sort_values("missing_count", ascending=False)
    )
    return missing_report


missing_report = validate_final_dataset(customer_churn)
display(missing_report)
print("Target distribution:")
display(
    customer_churn["churn"].value_counts()
    .rename(index={0: "No churn (auto-renew on)", 1: "Churn (auto-renew off)"})
    .to_frame("customer_count")
)
print("Final dataset checks passed.")

,missing_count,missing_percent
low_rating_share,8,3.62
average_review_rating,8,3.62
subscription_type_name,0,0.00
churn,0,0.00
subscription_age_days,0,0.00
order_count,0,0.00
total_payment_amount,0,0.00
delivery_service,0,0.00
customer_id,0,0.00
total_items_purchased,0,0.00


Target distribution:


,customer_count
churn,
No churn (auto-renew on),135
Churn (auto-renew off),86


Final dataset checks passed.


## 6. Load — write the final dataset as Parquet

**What this block does:** saves the transformed dataset in Parquet format. Parquet is a columnar format that preserves data types and is efficient for analytical workloads. PyArrow is used as the Parquet engine.

If PyArrow is not installed in your VS Code environment, run `pip install pyarrow` in the project terminal.

In [13]:
def load_parquet(frame: pd.DataFrame, output_path: Path) -> Path:
    """Write the final customer churn dataset to a Parquet file/directory."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(output_path, index=False, engine="pyarrow")
    logger.info("Saved Parquet dataset to %s", output_path)
    return output_path


saved_path = load_parquet(customer_churn, OUTPUT_PATH)
print("Saved:", saved_path)
print("File size (KB):", round(saved_path.stat().st_size / 1024, 2))

2026-11-22 01:26:24,430 | INFO | Saved Parquet dataset to c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project\data\processed\customer_churn.parquet


Saved: c:\Users\baind\Downloads\PerfectPetClub_Churn_Project (1)\PerfectPetClub_Churn_Project\data\processed\customer_churn.parquet
File size (KB): 34.02


## 7. Read-back test — verify the saved Parquet output

**What this block does:** reads the Parquet file back into Pandas and compares its shape, columns and values with the in-memory dataset. This checks that the load step produced a readable file and did not change the dataset unexpectedly.

In [14]:
reloaded = pd.read_parquet(OUTPUT_PATH, engine="pyarrow")

assert reloaded.shape == customer_churn.shape, "Parquet shape differs from transformed data."
assert list(reloaded.columns) == list(customer_churn.columns), "Parquet columns differ."
pd.testing.assert_frame_equal(
    reloaded.reset_index(drop=True),
    customer_churn.reset_index(drop=True),
    check_dtype=False
)

print("Read-back validation passed.")
print("Parquet rows:", len(reloaded))
print("Parquet columns:", len(reloaded.columns))
display(reloaded.head())

Read-back validation passed.
Parquet rows: 221
Parquet columns: 25


,customer_id,churn,subscription_type_name,delivery_service,subscription_age_days,order_count,total_payment_amount,average_payment_amount,total_items_purchased,average_items_per_order,...,crm_query_count,crm_channel_count,login_count,active_login_days,unique_sessions,mobile_login_count,desktop_login_count,nps_recommend,satisfaction_score,online_comparison_freq
0,1,1,SuperSaver,Economy72,313.728698,20,74455,3722.750000,46,2.300000,...,1,2.0,48,11,48,0,0,8,6,5
1,2,0,SuperSaver,Economy72,313.728698,253,1036961,4098.660079,748,2.956522,...,24,3.0,1085,295,1085,0,0,5,8,9
2,3,0,Premium,SignedFor24,313.728698,304,1284692,4225.960526,824,2.710526,...,77,3.0,624,266,624,0,0,9,5,5
3,4,0,Premium,SignedFor24,313.728698,1055,9577504,9078.202844,7362,6.978199,...,149,3.0,1540,309,1540,0,0,9,3,5
4,5,1,SuperSaver,Economy72,313.728698,44,179738,4084.954545,116,2.636364,...,4,3.0,388,99,388,388,0,0,8,10


## 8. Final feature dictionary and handover to the ML notebook

| Column | Meaning | Modelling guidance |
|---|---|---|
| `customer_id` | Internal customer key | Drop before training |
| `churn` | 1 = auto-renew off; 0 = auto-renew on | Target label |
| `subscription_type_name` | Basic/Premium/SuperSaver category | Categorical feature |
| `delivery_service` | Subscription delivery service | Categorical feature |
| `subscription_age_days` | Days from subscription start to latest observed activity date | Numeric feature |
| `order_count` | Number of orders | Numeric feature |
| `total_payment_amount` | Sum of recorded payment amounts | Numeric feature; source unit not specified |
| `average_payment_amount` | Average recorded payment amount | Numeric feature |
| `total_items_purchased` | Total item quantity across orders | Numeric feature |
| `average_items_per_order` | Average item quantity per order | Numeric feature |
| `review_count` | Number of product reviews | Numeric feature |
| `average_review_rating` | Mean star rating | Numeric feature |
| `low_rating_share` | Share of reviews rated 1–2 | Numeric feature |
| `crm_contact_count` | Number of customer service contacts | Numeric feature |
| `crm_complaint_count` | Number of complaint contacts | Numeric feature |
| `crm_query_count` | Number of query contacts | Numeric feature |
| `crm_channel_count` | Number of distinct CRM channels | Numeric feature |
| `login_count` | Number of login records | Numeric feature |
| `active_login_days` | Number of distinct login dates | Numeric feature |
| `unique_sessions` | Number of unique session IDs | Numeric feature |
| `mobile_login_count` | Login records with mobile user-agent marker | Numeric feature |
| `desktop_login_count` | Login records with desktop/laptop marker | Numeric feature |
| `nps_recommend` | Survey recommendation score (0–10) | Numeric feature |
| `satisfaction_score` | Survey satisfaction score (0–10) | Numeric feature |
| `online_comparison_freq` | Survey online-comparison frequency score (0–10) | Numeric feature |

### Handover notes for Task 3 (PySpark ML)

1. Read `data/processed/customer_churn.parquet` with Spark.
2. Separate `churn` as the label and exclude `customer_id`.
3. Impute missing numeric survey/average-rating fields using a fitted imputer.
4. Encode categorical subscription fields and assemble features using Spark ML transformers.
5. Split training and test data with a fixed seed and use stratification if implementing it explicitly.
6. Fit multiple classifiers and tune parameters using cross-validation on the training data only.
7. Report confusion matrix, precision, recall, F1-score and ROC-AUC where supported.
8. Because the target is imbalanced, compare churn-class recall and F1 rather than relying on accuracy alone.

### Limitations to discuss in the live assessment

- `auto_renew` is a current status flag; no cancellation-event timestamp is supplied.
- Activity features are aggregated across the supplied observation window, so they may include activity after a customer decided not to renew. This limits claims about future prediction.
- The assignment notes the data is synthetic and demographic patterns were not intentionally generated.
- Some customer records may have no CRM contacts or reviews; these are represented as zero counts or missing averages, respectively.
- Payment `amount` units are not documented in the database schema, so the feature is named as a raw amount rather than assuming a currency.
- The survey-question reference repeats recommendation-style wording for `satisfaction_score`; confirm this ambiguity with the module team if needed.

**ETL completion criterion:** the notebook runs from top to bottom, creates `data/processed/customer_churn.parquet`, and passes the read-back test.